# 05 · Feature engineering

Input: `data/interim/lendingclub_clean.parquet`
Output: `data/processed/train.parquet`, `data/processed/valid.parquet`, `data/processed/feature_config.json`

Scope: application-time features, leakage guard, chronological split, missingness audit. Imputation and encoding are fitted inside the modelling pipeline on training rows only; the datasets saved here keep original missing values.

In [9]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.3f}'.format)

TARGET = 'default'
SPLIT_KEY = 'issue_d'
TRAIN_END = pd.Timestamp('2014-01-01')
VALID_END = pd.Timestamp('2015-01-01')

INTERIM_PATH = Path('../data/interim/lendingclub_clean.parquet')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INTERIM_PATH)
print(f'Shape: {df.shape}')
print(f'Date range: {df[SPLIT_KEY].min():%b %Y} to {df[SPLIT_KEY].max():%b %Y}')
print(f'Default rate: {df[TARGET].mean():.2%}')

Shape: (453809, 73)
Date range: Jun 2007 to Dec 2014
Default rate: 17.03%


## 1. Leakage guard

In [10]:
POST_ORIGINATION = {
    'loan_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'pymnt_plan', 'hardship_flag', 'debt_settlement_flag',
}
POST_ORIGINATION_PREFIXES = ('hardship_', 'settlement_', 'debt_settlement')

leaky = sorted(
    c for c in df.columns
    if c != TARGET and (c in POST_ORIGINATION or c.startswith(POST_ORIGINATION_PREFIXES))
)
print('Post-origination columns removed:', leaky or 'none')
model_df = df.drop(columns=leaky)

Post-origination columns removed: none


## 2. Feature construction

In [11]:
assert model_df[SPLIT_KEY].notna().all(), 'Every loan needs an issue date'
known = model_df['earliest_cr_line'].notna()
assert (model_df.loc[known, 'earliest_cr_line'] <= model_df.loc[known, SPLIT_KEY]).all()
assert (model_df['loan_amnt'] > 0).all()

model_df['credit_hist_years'] = (
    (model_df[SPLIT_KEY] - model_df['earliest_cr_line']).dt.days / 365.25
)
model_df['loan_to_income'] = model_df['loan_amnt'] / model_df['annual_inc'].where(model_df['annual_inc'] > 0)

REDUNDANT = {
    'earliest_cr_line', 'fico_range_high', 'num_sats',
    'num_rev_tl_bal_gt_0', 'tot_cur_bal',
}
model_df = model_df.drop(columns=sorted(REDUNDANT & set(model_df.columns)))

bool_cols = model_df.drop(columns=TARGET).select_dtypes('bool').columns
if len(bool_cols):
    model_df[bool_cols] = model_df[bool_cols].astype('int8')
model_df[TARGET] = model_df[TARGET].astype('int8')

FEATURES = [c for c in model_df.columns if c not in {TARGET, SPLIT_KEY}]

assert TARGET not in FEATURES and SPLIT_KEY not in FEATURES
assert not set(FEATURES) & POST_ORIGINATION
assert not model_df[FEATURES].select_dtypes(include=['datetime', 'timedelta']).columns.tolist()
assert np.isfinite(model_df[['credit_hist_years', 'loan_to_income']].dropna()).all().all()

print(f'Features: {len(FEATURES)}')

Features: 68


In [12]:
HISTORY_COLS = ['avg_cur_bal', 'num_il_tl']

model_df['history_block_flag'] = model_df[HISTORY_COLS].isna().all(axis=1).astype('int8')
model_df[HISTORY_COLS] = model_df[HISTORY_COLS].fillna(-1)

## 3. Chronological split

Train: loans issued before 2014. Validation: loans issued in 2014. Treat 2014 as the final holdout: model and missingness-treatment choices are made with time-ordered cross-validation inside the training years.

In [13]:
year = model_df[SPLIT_KEY].dt.year
train_mask = model_df[SPLIT_KEY] < TRAIN_END
valid_mask = (model_df[SPLIT_KEY] >= TRAIN_END) & (model_df[SPLIT_KEY] < VALID_END)

assert train_mask.any() and valid_mask.any()
assert (train_mask | valid_mask).all(), 'Rows outside the split window'
assert not (train_mask & valid_mask).any()

train_df = model_df.loc[train_mask].reset_index(drop=True)
valid_df = model_df.loc[valid_mask].reset_index(drop=True)

cohort = model_df.groupby(year).agg(rows=(TARGET, 'size'), default_rate=(TARGET, 'mean'))
if 'term' in model_df.columns:
    cohort['share_60m'] = model_df['term'].eq(60).groupby(year).mean()
display(cohort)

split_summary = pd.DataFrame({
    'rows': [len(train_df), len(valid_df)],
    'start': [train_df[SPLIT_KEY].min(), valid_df[SPLIT_KEY].min()],
    'end': [train_df[SPLIT_KEY].max(), valid_df[SPLIT_KEY].max()],
    'default_rate': [train_df[TARGET].mean(), valid_df[TARGET].mean()],
}, index=['train', 'valid'])
split_summary

,rows,default_rate,share_60m
issue_d,,,
2007,603,0.262,0.000
2008,2393,0.207,0.000
2009,5281,0.137,0.000
2010,12537,0.140,0.270
2011,21721,0.152,0.351
2012,53367,0.162,0.185
2013,134804,0.156,0.255
2014,223103,0.184,0.271


,rows,start,end,default_rate
train,230706,2007-06-01,2013-12-01,0.156
valid,223103,2014-01-01,2014-12-01,0.184


## 4. Missingness audit

In [14]:
INDICATOR_COLS = [c for c in FEATURES if c.endswith('_missing')]
for ind in INDICATOR_COLS:
    base = ind.removesuffix('_missing')
    assert base in FEATURES, f'{ind} has no source column'
    for name, part in (('train', train_df), ('valid', valid_df)):
        same = part[ind].astype('int8').eq(part[base].isna().astype('int8')).all()
        assert same, f'{ind} does not match {base} missingness in {name}'

audit = pd.DataFrame({
    'train_missing_%': train_df[FEATURES].isna().mean() * 100,
    'valid_missing_%': valid_df[FEATURES].isna().mean() * 100,
})
audit = audit[audit['train_missing_%'] > 0].sort_values('train_missing_%', ascending=False)
audit['has_indicator'] = [f'{c}_missing' in FEATURES for c in audit.index]
audit

,train_missing_%,valid_missing_%,has_indicator
mths_since_last_record,90.844,82.089,True
mths_since_last_major_derog,85.907,71.532,True
mths_since_recent_bc_dlq,84.072,73.505,True
mths_since_recent_revol_delinq,76.402,63.997,True
mths_since_last_delinq,58.297,49.146,True
mo_sin_old_il_acct,33.119,3.093,False
num_tl_120dpd_2m,30.572,3.435,False
pct_tl_nvr_dlq,30.528,0.000,False
mths_since_recent_inq,30.516,9.047,False
mo_sin_rcnt_rev_tl_op,30.462,0.000,False


In [15]:
high_missing = audit.index[audit['train_missing_%'] >= 15].tolist()
(
    train_df[high_missing].isna()
    .groupby(train_df[SPLIT_KEY].dt.year).mean().mul(100).T
)

issue_d,2007,2008,2009,2010,2011,2012,2013
mths_since_last_record,6.965,60.635,95.209,94.002,94.738,97.238,88.132
mths_since_last_major_derog,100.000,100.000,100.000,100.000,100.000,89.913,79.874
mths_since_recent_bc_dlq,100.000,100.000,100.000,100.000,100.000,87.429,77.717
mths_since_recent_revol_delinq,100.000,100.000,100.000,100.000,100.000,73.924,69.936
mths_since_last_delinq,6.468,38.069,66.957,64.282,66.208,58.840,56.502
mo_sin_old_il_acct,100.000,100.000,100.000,100.000,100.000,54.185,3.676
num_tl_120dpd_2m,100.000,100.000,100.000,100.000,100.000,51.982,0.190
pct_tl_nvr_dlq,100.000,100.000,100.000,100.000,100.000,51.982,0.113
mths_since_recent_inq,100.000,100.000,100.000,100.000,100.000,24.896,10.816
mo_sin_rcnt_rev_tl_op,100.000,100.000,100.000,100.000,100.000,51.983,0.000


## 5. Column groups and preprocessing check

Groups are written to `feature_config.json` so the modelling pipeline builds the same preprocessor. Event-history fields (`mths_since_*` with an indicator) are kept as a separate group: a missing value means no event on record or not collected in that period, so a median fill would invent an event.

In [16]:
X_train, X_valid = train_df[FEATURES], valid_df[FEATURES]

numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
FLAGGED_COLS = [c.removesuffix('_missing') for c in INDICATOR_COLS]
EVENT_HISTORY_COLS = [c for c in FLAGGED_COLS if c.startswith('mths_since_')]
MODE_COLS = [c for c in ['emp_length'] if c in numeric_cols]
MEDIAN_COLS = [c for c in numeric_cols if c not in set(MODE_COLS) | set(EVENT_HISTORY_COLS)]
CATEGORICAL_COLS = [c for c in FEATURES if c not in numeric_cols]

assert set(EVENT_HISTORY_COLS) <= set(numeric_cols)
assert sorted(MEDIAN_COLS + MODE_COLS + EVENT_HISTORY_COLS + CATEGORICAL_COLS) == sorted(FEATURES)

GROUPS = {
    'numeric_median': MEDIAN_COLS,
    'numeric_mode': MODE_COLS,
    'event_history': EVENT_HISTORY_COLS,
    'categorical': CATEGORICAL_COLS,
}


def build_preprocessor(groups):
    return ColumnTransformer(
        [
            ('numeric_median', SimpleImputer(strategy='median', keep_empty_features=True), groups['numeric_median']),
            ('numeric_mode', SimpleImputer(strategy='most_frequent', keep_empty_features=True), groups['numeric_mode']),
            ('event_history', SimpleImputer(strategy='constant', fill_value=-1), groups['event_history']),
            ('categorical', SimpleImputer(strategy='constant', fill_value='Missing'), groups['categorical']),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    ).set_output(transform='pandas')


preprocessor = build_preprocessor(GROUPS)
X_train_pp = preprocessor.fit_transform(X_train)
X_valid_pp = preprocessor.transform(X_valid)

assert list(X_train_pp.columns) == list(X_valid_pp.columns)
assert set(X_train_pp.columns) == set(FEATURES), 'A feature was lost or added by the preprocessor'
assert X_train_pp.isna().sum().sum() == 0 and X_valid_pp.isna().sum().sum() == 0
assert set(X_train_pp.select_dtypes(exclude='number').columns) == set(CATEGORICAL_COLS)

print('Preprocessed shapes:', X_train_pp.shape, X_valid_pp.shape)
print({name: len(cols) for name, cols in GROUPS.items()})

Preprocessed shapes: (230706, 68) (223103, 68)
{'numeric_median': 57, 'numeric_mode': 1, 'event_history': 5, 'categorical': 5}


## 6. Save to `data/processed`

In [17]:
COLUMNS = [SPLIT_KEY, TARGET] + FEATURES

train_df[COLUMNS].to_parquet(PROCESSED_DIR / 'train.parquet', index=False)
valid_df[COLUMNS].to_parquet(PROCESSED_DIR / 'valid.parquet', index=False)

config = {
    'target': TARGET,
    'split_key': SPLIT_KEY,
    'train_end': TRAIN_END.date().isoformat(),
    'valid_end': VALID_END.date().isoformat(),
    'features': FEATURES,
    **GROUPS,
    'missing_indicators': INDICATOR_COLS,
    'rows': {'train': len(train_df), 'valid': len(valid_df)},
}
(PROCESSED_DIR / 'feature_config.json').write_text(json.dumps(config, indent=2))

for name, frame in (('train', train_df), ('valid', valid_df)):
    back = pd.read_parquet(PROCESSED_DIR / f'{name}.parquet')
    assert list(back.columns) == COLUMNS
    assert back.shape == (len(frame), len(COLUMNS))
    assert back[TARGET].equals(frame[TARGET])
    print(f'{name}.parquet  {back.shape}  default rate {back[TARGET].mean():.2%}')

train.parquet  (230706, 70)  default rate 15.65%
valid.parquet  (223103, 70)  default rate 18.45%


### Missing-Value Handling Strategy

I handle missing values according to the column type, missingness rate, and meaning of the missing data.

| Group | Columns | Missingness | Fill Strategy |
|---|---|---|---|
| `numeric_median` | All other numeric columns | 0.01% to approximately 30% | Train-set median |
| `numeric_mode` | `emp_length` | Approximately 6.5% | Train-set most-frequent value, preserving its ordinal categories |
| `event_history` | `mths_since_*` columns that already have a corresponding `_missing` flag | Approximately 58% to 91% | Fill with `-1` and retain the existing missingness flag |
| `categorical` | Text/categorical columns | Any missingness rate | Introduce a new `"Missing"` category |

**Rationale**

- **Numeric columns:** Median imputation provides a robust baseline that is less sensitive to outliers than mean imputation. The median must be learned from the training set only and then applied to validation and test data.
- **`emp_length`:** Use the most-frequent training-set value as a simple baseline. Although employment length is ordinal, mode imputation preserves a valid category rather than introducing an artificial value.
- **Event-history columns:** A value of `-1` acts as a sentinel for missing historical information, while the existing `_missing` flag explicitly identifies missingness. This allows the model to distinguish unavailable history from observed values. Ensure that `-1` cannot be confused with a valid value.
- **Categorical columns:** The `"Missing"` category preserves missingness as a potentially informative signal instead of replacing it with an existing category.

**Implementation note:** Fit all imputation values using the training data only. Apply the learned values consistently to validation, test, and future data to prevent data leakage.